# 05 — Mixed-model association: DRVI factors × CPS

Fits one mixed linear model per (factor, CPS phenotype) pair with a **donor random intercept**.
Two nested models are compared for each factor:

- **pan**: one shared factor slope across all regions (+ region fixed effects)
- **regional**: separate factor slope per region

A likelihood-ratio test between them flags factors whose association with a CPS score
differs across brain regions.

**Units**: donor × brain-region pseudobulk (~530 rows) rather than individual nuclei,
because CPS_Local is constant within a donor × region group.

**Factors**: the 64 DRVI latent dimensions are ReLU-split into 128 half-rectified directions
(factor_{2i−1} = relu(DR_i), factor_{2i} = relu(−DR_i)) so that positive and negative
loadings are tested separately.

**Covariates**: sex, age, and PMI at donor level.
`library_prep` (902 levels, per-library) does not aggregate to a single value per
donor × region and is excluded.

**Data**: pseudobulk loaded from `03_pseudobulk.ipynb` output (run that notebook first).

## 1 — Setup

In [1]:
from __future__ import annotations

import warnings

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import chi2
from statsmodels.stats.multitest import multipletests

import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import sst_drvi as S

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_rows", 140)

PSEUDO_DIR = S.SCRATCH / "pseudobulk"

### Data

Loaded from `03_pseudobulk.ipynb` output.

In [ ]:
dat = pd.read_parquet(PSEUDO_DIR / "dat.parquet")
factor_columns = sorted(
    [c for c in dat.columns if c.startswith("factor_")],
    key=lambda x: int(x.split("_")[1]),
)
print(
    f"{len(dat):,} donor×region units, "
    f"{dat['donor'].nunique()} donors, "
    f"{dat['region'].nunique()} regions, "
    f"{len(factor_columns)} factor directions"
)

## 2 — Model

`test_factor` fits two nested MixedLM models (ML, not REML) and compares them
with a likelihood-ratio test:

- **pan**: one shared factor slope across all regions (+ region fixed effects)
- **regional**: a separate factor slope per region

The LRT χ²(9) tests whether the regional slopes are meaningfully heterogeneous.
Within each region the factor is z-scored (one-SD change has the same meaning
everywhere). Results are BH-corrected separately within each CPS phenotype.

In [4]:
def test_factor(
    data: pd.DataFrame,
    factor: str,
    phenotype: str,
    covariates: list[str],
) -> tuple[dict, pd.DataFrame]:
    df = data[
        ["donor", "region", phenotype, factor, *covariates]
    ].dropna().copy()

    # A one-SD change has the same meaning in every region.
    df["factor_z"] = df.groupby("region")[factor].transform(
        lambda x: (x - x.mean()) / x.std(ddof=0)
    )

    covariate_formula = (
        " + " + " + ".join(covariates) if covariates else ""
    )

    # One shared factor slope across all regions.
    pan_formula = (
        f"{phenotype} ~ factor_z + C(region)"
        f"{covariate_formula}"
    )
    pan_fit = smf.mixedlm(
        pan_formula,
        data=df,
        groups=df["donor"],
    ).fit(reml=False, method="lbfgs")

    # Separate factor slope for every region.
    regional_formula = (
        f"{phenotype} ~ 0 + C(region) "
        f"+ C(region):factor_z"
        f"{covariate_formula}"
    )
    regional_fit = smf.mixedlm(
        regional_formula,
        data=df,
        groups=df["donor"],
    ).fit(reml=False, method="lbfgs")

    # Likelihood-ratio test: do factor slopes differ across regions?
    lr_stat = max(0.0, 2 * (regional_fit.llf - pan_fit.llf))
    lr_df = len(regional_fit.fe_params) - len(pan_fit.fe_params)
    heterogeneity_p = chi2.sf(lr_stat, lr_df)

    overall = {
        "factor": factor,
        "n_observations": len(df),
        "n_donors": df["donor"].nunique(),
        "pan_beta": pan_fit.fe_params["factor_z"],
        "pan_se": pan_fit.bse_fe["factor_z"],
        "pan_p": pan_fit.pvalues["factor_z"],
        "heterogeneity_lr": lr_stat,
        "heterogeneity_df": lr_df,
        "heterogeneity_p": heterogeneity_p,
        "pan_converged": pan_fit.converged,
        "regional_converged": regional_fit.converged,
    }

    rows = []
    for term in regional_fit.fe_params.index:
        if term.endswith(":factor_z"):
            # Example term: C(region)[Heart]:factor_z
            region = term.split("[", maxsplit=1)[1].split("]", maxsplit=1)[0]

            beta = regional_fit.fe_params[term]
            se = regional_fit.bse_fe[term]

            rows.append(
                {
                    "factor": factor,
                    "region": region,
                    "beta": beta,
                    "se": se,
                    "ci_low": beta - 1.96 * se,
                    "ci_high": beta + 1.96 * se,
                    "p": regional_fit.pvalues[term],
                }
            )

    return overall, pd.DataFrame(rows)

## 3 — Fit

BH correction is applied per phenotype × testing family:
- `pan_fdr`: over all 128 pan p-values
- `heterogeneity_fdr`: over all 128 LRT p-values
- `regional_fdr`: over all factor × region p-values

In [6]:
# Three local CPS phenotypes (see 04_progression.ipynb).
LOCAL_SCORES = {
    "CPS_Local": "composite",
    "CPS_Local_ABeta": "ABeta",
    "CPS_Local_pTau": "pTau",
}

# Sex, age, and PMI are available at donor level; library_prep excluded (per-library).
covariates = ["age", "sex", "PMI"]

all_overall, all_regional = [], []

for phenotype, label in LOCAL_SCORES.items():
    print(f"\n=== {phenotype} ({label}) — {len(factor_columns)} factors ===")

    ph_overall, ph_regional = [], []
    for factor in factor_columns:
        overall, regional = test_factor(
            data=dat,
            factor=factor,
            phenotype=phenotype,
            covariates=covariates,
        )
        overall["phenotype"] = phenotype
        regional["phenotype"] = phenotype
        ph_overall.append(overall)
        ph_regional.append(regional)

    # Correct the three families separately within this phenotype.
    ph_df = pd.DataFrame(ph_overall)
    ph_df["pan_fdr"] = multipletests(ph_df["pan_p"], method="fdr_bh")[1]
    ph_df["heterogeneity_fdr"] = multipletests(ph_df["heterogeneity_p"], method="fdr_bh")[1]

    ph_reg = pd.concat(ph_regional, ignore_index=True)
    ph_reg["regional_fdr"] = multipletests(ph_reg["p"], method="fdr_bh")[1]

    n_pan = (ph_df["pan_fdr"] < 0.05).sum()
    n_het = (ph_df["heterogeneity_fdr"] < 0.05).sum()
    n_reg = (ph_reg["regional_fdr"] < 0.05).sum()
    print(
        f"  pan FDR<0.05: {n_pan}  |  "
        f"heterogeneous FDR<0.05: {n_het}  |  "
        f"regional FDR<0.05: {n_reg}"
    )

    all_overall.append(ph_df)
    all_regional.append(ph_reg)

overall_results = pd.concat(all_overall, ignore_index=True)
regional_results = pd.concat(all_regional, ignore_index=True)


=== CPS_Local (composite) — 128 factors ===
  pan FDR<0.05: 0  |  heterogeneous FDR<0.05: 0  |  regional FDR<0.05: 0

=== CPS_Local_ABeta (ABeta) — 128 factors ===
  pan FDR<0.05: 0  |  heterogeneous FDR<0.05: 17  |  regional FDR<0.05: 0

=== CPS_Local_pTau (pTau) — 128 factors ===
  pan FDR<0.05: 0  |  heterogeneous FDR<0.05: 0  |  regional FDR<0.05: 0


## 4 — Results

Reading the tests:
- `pan_fdr < 0.05`, heterogeneity not significant → shared pan-region association
- `heterogeneity_fdr < 0.05` → effect size varies among regions
- `regional_fdr < 0.05` → evidence for an association in that specific region
- Both pan and heterogeneity significant → broadly associated, effect size varies

In [7]:
# Convergence summary.
n_pan_fail = (~overall_results["pan_converged"]).sum()
n_reg_fail = (~overall_results["regional_converged"]).sum()
print(f"Pan model non-converged: {n_pan_fail} / {len(overall_results)}")
print(f"Regional model non-converged: {n_reg_fail} / {len(overall_results)}")

Pan model non-converged: 0 / 384
Regional model non-converged: 0 / 384


In [8]:
# Pan-region hits: shared slope, FDR<0.05, heterogeneity not significant.
pan_hits = overall_results[
    (overall_results["pan_fdr"] < 0.05)
    & ~(overall_results["heterogeneity_fdr"] < 0.05)
].sort_values(["phenotype", "pan_fdr"])

print(f"Pan-region hits: {len(pan_hits)}")
pan_hits[
    ["phenotype", "factor", "pan_beta", "pan_se", "pan_p", "pan_fdr", "n_donors", "pan_converged"]
]

Pan-region hits: 0


,phenotype,factor,pan_beta,pan_se,pan_p,pan_fdr,n_donors,pan_converged


In [9]:
# Heterogeneous hits: effect size varies by region.
het_hits = overall_results[
    overall_results["heterogeneity_fdr"] < 0.05
].sort_values(["phenotype", "heterogeneity_fdr"])

print(f"Heterogeneous hits: {len(het_hits)}")
het_hits[
    ["phenotype", "factor", "pan_beta", "pan_fdr", "heterogeneity_lr", "heterogeneity_fdr", "regional_converged"]
]

Heterogeneous hits: 17


,phenotype,factor,pan_beta,pan_fdr,heterogeneity_lr,heterogeneity_fdr,regional_converged
252,CPS_Local_ABeta,factor_125,-0.010624,0.633447,36.265184,0.004553,True
135,CPS_Local_ABeta,factor_8,-0.005801,0.860138,30.450313,0.010951,True
153,CPS_Local_ABeta,factor_26,0.006768,0.783918,31.950603,0.010951,True
185,CPS_Local_ABeta,factor_58,0.006909,0.860138,30.605596,0.010951,True
206,CPS_Local_ABeta,factor_79,-0.003209,0.874280,30.064398,0.010951,True
174,CPS_Local_ABeta,factor_47,-0.007674,0.638953,29.512509,0.011325,True
168,CPS_Local_ABeta,factor_41,-0.002843,0.898645,27.683952,0.019694,True
141,CPS_Local_ABeta,factor_14,0.004550,0.860679,26.415088,0.027946,True
131,CPS_Local_ABeta,factor_4,-0.002559,0.906001,25.783963,0.031516,True
197,CPS_Local_ABeta,factor_70,-0.009229,0.579102,25.450290,0.032146,True


## 5 — Regional patterns

In [10]:
# Region-specific hits.
reg_hits = regional_results[
    regional_results["regional_fdr"] < 0.05
].sort_values(["phenotype", "regional_fdr"])

print(f"Region-specific hits: {len(reg_hits)}")
reg_hits[
    ["phenotype", "factor", "region", "beta", "se", "ci_low", "ci_high", "p", "regional_fdr"]
]

Region-specific hits: 0


,phenotype,factor,region,beta,se,ci_low,ci_high,p,regional_fdr


## 6 — Top hits

For the heterogeneous factors (those whose association with CPS differs
significantly across regions), plot the per-region slopes from the regional model.
The dashed line is the pan-region slope from the simpler model.

In [ ]:
# Forest plot: per-region slopes for the heterogeneous hits.
# Each panel is one factor × phenotype pair; bars are 95% Wald CIs from the regional model.
top_het = het_hits.sort_values("heterogeneity_fdr").head(6)

if len(top_het) == 0:
    print("No heterogeneous hits to plot.")
else:
    import matplotlib.pyplot as plt
    import numpy as np

    n_panels = min(6, len(top_het))
    ncols = min(3, n_panels)
    nrows = (n_panels + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows), sharey=False)
    axes = np.array(axes).flatten()

    for ax, (_, row) in zip(axes, top_het.iterrows()):
        factor    = row["factor"]
        phenotype = row["phenotype"]
        reg = regional_results[
            (regional_results["factor"] == factor) &
            (regional_results["phenotype"] == phenotype)
        ].sort_values("beta").reset_index(drop=True)

        y = np.arange(len(reg))
        ax.errorbar(
            reg["beta"], y,
            xerr=[reg["beta"] - reg["ci_low"], reg["ci_high"] - reg["beta"]],
            fmt="o", color="steelblue", capsize=3, markersize=5, lw=1.2,
        )
        ax.axvline(row["pan_beta"], color="tomato", ls="--", lw=1,
                   label=f"pan={row['pan_beta']:.3f}")
        ax.axvline(0, color="k", lw=0.5, ls=":")
        ax.set(yticks=y, yticklabels=reg["region"].tolist(),
               xlabel="regional slope (95% CI)",
               title=f"{factor}\n{phenotype.replace('CPS_Local_', 'CPS Local ')}")
        ax.tick_params(axis="y", labelsize=7)
        ax.legend(fontsize=7)

    for ax in axes[n_panels:]:
        ax.set_visible(False)

    plt.suptitle("Per-region slopes for heterogeneous hits", y=1.01)
    plt.tight_layout()
    plt.show()

## 7 — Pathology specificity

`CPS_Local` is a composite of ABeta and pTau. The two components are correlated
but not interchangeable; a direction can favour one. The fits below compare the
slopes from the three phenotypes so that the *gap* between ABeta and pTau slopes
carries the specificity claim — neither column alone is sufficient.

Read the gap, not either slope: a direction driven by tau will still show a
substantial ABeta slope, because the two scores correlate.

In [ ]:
# Pivot to one row per direction, columns per phenotype.
LABEL = {"CPS_Local": "composite", "CPS_Local_ABeta": "ABeta", "CPS_Local_pTau": "pTau"}
patho = (
    overall_results
    .assign(label=overall_results["phenotype"].map(LABEL))
    .pivot(index="factor", columns="label", values=["pan_beta", "pan_fdr"])
)
patho.columns = [f"{label}_{col.replace('pan_', '')}" for col, label in patho.columns]
patho["gap"] = patho["ABeta_beta"] - patho["pTau_beta"]
patho["best_fdr"] = patho[["composite_fdr", "ABeta_fdr", "pTau_fdr"]].min(axis=1)

Q_CUT = 0.05
specific = patho[patho["best_fdr"] < Q_CUT]
print(f"Directions with best FDR < {Q_CUT}: {len(specific)}")
print("Most component-specific (largest gap) among those:")
specific.reindex(specific["gap"].abs().sort_values(ascending=False).index)[
    ["composite_beta", "ABeta_beta", "pTau_beta", "gap", "best_fdr"]
].head(12).round(3)

The two components are far from interchangeable, so a direction can plausibly
favour one. They are not independent either: a direction genuinely driven by
ABeta will still show a substantial pTau slope. Only the *gap* between the two
carries the specificity claim; read the difference, never either column alone.

In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 5.0))
lim = 1.08 * patho[["ABeta_beta", "pTau_beta"]].abs().to_numpy().max()
ax.plot([-lim, lim], [-lim, lim], ls="--", color="0.75", lw=0.9, zorder=0)
ax.axhline(0, color="0.85", lw=0.8)
ax.axvline(0, color="0.85", lw=0.8)

hit = patho["best_fdr"] < Q_CUT
ax.scatter(patho.loc[~hit, "ABeta_beta"], patho.loc[~hit, "pTau_beta"],
           s=20, color="0.78", label=f"FDR ≥ {Q_CUT} on all")
ax.scatter(patho.loc[hit, "ABeta_beta"], patho.loc[hit, "pTau_beta"],
           s=42, c=patho.loc[hit, "composite_beta"], cmap="coolwarm",
           vmin=-0.7, vmax=0.7, edgecolor="white", linewidth=0.4,
           label=f"FDR < {Q_CUT} on ≥1 component")
for name in patho.loc[hit, "gap"].abs().nlargest(6).index:
    ax.annotate(name, (patho.loc[name, "ABeta_beta"], patho.loc[name, "pTau_beta"]),
                fontsize=7, xytext=(4, 3), textcoords="offset points")
ax.set(xlim=(-lim, lim), ylim=(-lim, lim),
       xlabel="pan slope vs CPS_Local_ABeta",
       ylabel="pan slope vs CPS_Local_pTau")
ax.set_title("Amyloid or tau?  (colour = composite slope)", fontsize=10)
ax.legend(fontsize=7, frameon=False, loc="upper left")
plt.tight_layout()

In [ ]:
leans_ptau = patho["pTau_beta"].abs() > patho["ABeta_beta"].abs()
signed_rank = stats.wilcoxon(patho["pTau_beta"].abs(), patho["ABeta_beta"].abs())
print(f"|pan slope| larger for pTau in {int(leans_ptau.sum())} of {len(patho)} directions "
      f"(Wilcoxon signed-rank p = {signed_rank.pvalue:.1e})")
print("median |slope|: " + ", ".join(
    f"{label} {patho[f'{label}_beta'].abs().median():.3f}"
    for label in ["composite", "ABeta", "pTau"]))

local = [c for c in overall_results.columns if c.startswith("CPS_Local")]
if len(local) == 0:
    local = ["CPS_Local", "CPS_Local_ABeta", "CPS_Local_pTau"]
scores = overall_results.groupby("phenotype")[local[0] if len(local) == 1 else "pan_beta"].first()
component = {"composite": "CPS_Local", "ABeta": "CPS_Local_ABeta", "pTau": "CPS_Local_pTau"}
print("\nFor reference: IQR of each CPS score across donor×region units:")
for label, col in component.items():
    vals = pd.to_numeric(dat[col], errors="coerce").dropna()
    print(f"  {label}: IQR = {vals.quantile(0.75) - vals.quantile(0.25):.3f}")
for label in ("pTau", "ABeta"):
    rho = stats.spearmanr(
        pd.to_numeric(dat["CPS_Local"], errors="coerce"),
        pd.to_numeric(dat[component[label]], errors="coerce"),
        nan_policy="omit",
    ).statistic
    print(f"  composite vs {label}: rho = {rho:.3f}")

The lean is cohort-wide, not a property of a few directions: most live directions
have a larger slope against tau than against amyloid, and the signed-rank test on
the paired magnitudes is not close. The honest reading is therefore about the
latent space as a whole — **these factors track tau burden more closely than
amyloid burden** — rather than about any single direction being tau-specific.

Two alternative explanations are printed above: a score with more spread across
units lifts every association with it, and the composite is itself nearer one
component, so a direction fit to track "progression" inherits some of that bias.

## 8 — Compositional context

`/data/multiregion/pertpy/CPS_Local/` holds a finished scCODA compositional
analysis of all 174 supertypes against `CPS_Local`. The cells below read it and
ask whether the directions that track CPS mark the supertypes scCODA says are
depleted. See the analysis README for the four caveats about how to use it.

In [ ]:
# SUPERTYPES: the SST supertype names used by scCODA.
# Loaded from the within-supertype pseudobulk written by 03_pseudobulk.
within_meta = pd.read_parquet(PSEUDO_DIR / "within_meta.parquet")
SUPERTYPES = within_meta.index.get_level_values("Supertype").unique().tolist()
print(f"{len(SUPERTYPES)} SST supertypes")

# Three values with three meanings, which a naive fillna(0) would collapse:
# NaN  -- supertype absent from that region, never modelled
# 0.0  -- modelled, but scCODA did not call the effect credible
# else -- a credible shift (negative = depleted)
summary_path = sorted(S.CPS_LOCAL_DIR.glob(S.SCCODA_SUMMARY_GLOB))[-1]
summary = pd.read_csv(summary_path, index_col=0)
effects = (summary[summary["Cell Type"].isin(SUPERTYPES)]
           .pivot(index="Cell Type", columns="Region", values="Local Model"))
effects.columns.name = None
print(f"{summary_path.name}: {effects.shape[0]} supertypes x {effects.shape[1]} regions -- "
      f"{int((effects.fillna(0) != 0).to_numpy().sum())} credible, "
      f"{int((effects == 0).to_numpy().sum())} not credible, "
      f"{int(effects.isna().to_numpy().sum())} not modelled")
effects.round(2)

### That table has ten columns and does not have ten measurements

The shape invites a supertype × region heatmap. Before drawing one: independent
per-region MCMC fits cannot agree to six decimal places, and across the SST
supertypes this table's effect is a single value repeated over every region the
supertype appears in — MTG being the one exception. Reading it as ten regional
measurements would multiply the evidence by ten.

In [ ]:
rows = {}
for supertype, values in effects.iterrows():
    present = values.dropna()
    if present.empty:
        continue
    mode = present.mode()
    shared = float(mode.iloc[0]) if len(mode) else np.nan
    rows[supertype] = {
        "n_regions": len(present),
        "n_distinct": int(present.nunique()),
        "shared_effect": shared,
        "deviating_regions": ", ".join(present.index[present != shared]) or "-",
    }
specificity = pd.DataFrame(rows).T.astype(
    {"n_regions": int, "n_distinct": int, "shared_effect": float})
specificity.index.name = "Cell Type"
print(f"{int((specificity['n_distinct'] == 1).sum())} of {len(specificity)} supertypes "
      "carry one value across every region they appear in")
specificity

So the usable scCODA quantity for the SST cohort is **one CPS_Local abundance
effect per supertype** (`shared_effect`), not a regional map. Significance is the
posterior inclusion probability, not a non-zero `Final Parameter`.

In [ ]:
# Reads the reference-sweep to get inclusion probabilities and effect spread.
# The neuronal results files are ~180 MB total; this chunk-read is the fast path.
SWEEP_COLUMNS = ["Region", "Covariate", "Cell Type", "Final Parameter",
                 "Inclusion probability", "Reference Cell Type"]
chunks = []
for path in sorted(S.CPS_LOCAL_DIR.glob(S.SCCODA_RESULTS_GLOB)):
    for chunk in pd.read_csv(path, usecols=SWEEP_COLUMNS, chunksize=500_000):
        chunk = chunk[(chunk["Covariate"] == "CPS_Local")
                      & chunk["Cell Type"].isin(SUPERTYPES)]
        if len(chunk):
            chunks.append(chunk)
swept = pd.concat(chunks, ignore_index=True)
swept = swept[swept["Cell Type"] != swept["Reference Cell Type"]]
by_cr = swept.groupby(["Cell Type", "Region"], observed=True)
sweep = pd.DataFrame({
    "n_references": by_cr["Final Parameter"].size(),
    "median_effect": by_cr["Final Parameter"].median(),
    "effect_p05": by_cr["Final Parameter"].quantile(0.05),
    "effect_p95": by_cr["Final Parameter"].quantile(0.95),
    "median_inclusion": by_cr["Inclusion probability"].median(),
    "frac_credible": by_cr["Inclusion probability"].apply(
        lambda v: float((v >= S.SCCODA_INCLUSION_THRESHOLD).mean())),
})
global_fit = sweep.xs("Global", level="Region").sort_values("median_effect")
print(f"{len(sweep)} supertype x region cells from "
      f"{int(sweep['n_references'].median())} references each")

### The summary *is* the region-agnostic fit

The boundary for credibility runs between `Sst_9` (inclusion 0.815, zeroed) and
`Sst_22` (0.841, kept). The summary reproduces the sweep's Global estimate to
within 0.016 for every called effect.

In [ ]:
order = global_fit.index
fig, axes = plt.subplots(1, 2, figsize=(14.5, 0.42 * len(order) + 2.8),
                         gridspec_kw={"width_ratios": [1.25, 1]})
axes[0].set_facecolor("0.88")
sns.heatmap(effects.reindex(order), cmap="vlag", center=0, vmin=-2, vmax=2,
            annot=True, fmt=".2f", annot_kws={"fontsize": 6.5}, linewidths=0.3,
            cbar_kws={"label": "scCODA CPS_Local effect", "shrink": 0.6}, ax=axes[0])
axes[0].set(title="Region-specific: delivered summary\ngrey = absent", xlabel="", ylabel="")
credible = (global_fit["median_inclusion"] >= S.SCCODA_INCLUSION_THRESHOLD).to_numpy()
y = np.arange(len(order))
axes[1].barh(y, global_fit["median_effect"], height=0.66,
             color=np.where(credible, "#3b6ea5", "white"), edgecolor="#3b6ea5")
axes[1].hlines(y, global_fit["effect_p05"], global_fit["effect_p95"], color="0.25", lw=1.2)
for pos, (_, row) in enumerate(global_fit.iterrows()):
    left = row["median_effect"] < 0
    axes[1].text(row["median_effect"] + (-0.04 if left else 0.04), pos,
                 f"{row['median_inclusion']:.2f}", fontsize=7, va="center",
                 ha="right" if left else "left")
axes[1].axvline(0, color="0.6", lw=0.8)
axes[1].set(yticks=y, yticklabels=order, ylim=(len(order) - 0.5, -0.5),
            xlabel="effect on supertype share (median over references)")
axes[1].set_title(f"Region-agnostic: Global fit\n"
                  f"numbers = inclusion prob; hollow = below {S.SCCODA_INCLUSION_THRESHOLD}", fontsize=10)
plt.tight_layout()

### 8b — Direction link

Do the directions that track CPS also mark the depleted supertypes? Each direction
gets an x-coordinate from how strongly its per-supertype activation profile
correlates (Spearman) with the scCODA depletion effect, and a y-coordinate from
its CPS association slope.

In [ ]:
# Pan slopes for each direction (composite phenotype), indexed by factor name.
pan_composite = (
    overall_results[overall_results["phenotype"] == "CPS_Local"]
    .set_index("factor")[["pan_beta", "pan_fdr"]]
)

# Supertype mean pan slope: correlate each direction's pan slope with the scCODA
# depletion effect across supertypes.  No embed needed — use the within pseudobulk.
within_act = pd.read_parquet(PSEUDO_DIR / "within_activity.parquet")
profile = within_act.groupby(level="Supertype").mean()
profile = (profile - profile.mean()) / profile.std(ddof=0).mask(lambda s: s == 0)

vulnerability = specificity["shared_effect"].reindex(profile.index)
usable = vulnerability.notna()

link = pd.DataFrame({
    "profile_vs_vulnerability": profile[usable.to_numpy()].corrwith(
        vulnerability[usable], method="spearman"
    ),
    "pan_beta": pan_composite["pan_beta"],
    "pan_fdr": pan_composite["pan_fdr"],
}).dropna()
link["composition_marker"] = link["profile_vs_vulnerability"].abs() > 0.5
link.reindex(link["pan_beta"].abs().sort_values(ascending=False).index).head(15).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 4.6))
ax.axhline(0, color="0.8", lw=0.8)
ax.axvline(0, color="0.8", lw=0.8)
points = ax.scatter(
    link["profile_vs_vulnerability"], link["pan_beta"],
    c=-np.log10(link["pan_fdr"].clip(lower=1e-300)),
    cmap="viridis", s=36, edgecolor="white", linewidth=0.4,
)
for name in link["pan_beta"].abs().nlargest(5).index:
    ax.annotate(name,
                (link.loc[name, "profile_vs_vulnerability"], link.loc[name, "pan_beta"]),
                fontsize=7, xytext=(4, 3), textcoords="offset points")
fig.colorbar(points, ax=ax, label=r"$-\log_{10}$FDR, pan CPS_Local")
ax.set(xlabel=r"$\rho$(supertype profile, scCODA CPS effect)",
       ylabel="pan slope vs CPS_Local")
ax.set_title("Composition marker, or state change?", fontsize=10)
plt.tight_layout()